# EpigraNet Tamil OCR on Google Colab

This notebook runs the existing EpigraNet OCR inference pipeline in Google Colab.
It uses Hugging Face Hub as the model registry for the checkpoint, reference embeddings,
and class-mapping assets.

Before running the notebook:

1. Push this project to GitHub.
2. Upload these runtime assets to a Hugging Face model repository:
   - `epigranet_embedding_model (1).pt`
   - `reference_embeddings.pt`
   - `class_mapping_209 (1).json`
3. Update `REPO_URL` and `HF_REPO_ID` in the config cell below.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/<your-github-user>/epirgranet_implementation_full.git"
PROJECT_DIR = Path("/content/epirgranet_implementation_full")

HF_REPO_ID = "your-hf-username/epigranet-tamil-ocr"
HF_REVISION = "main"
HF_TOKEN = None  # Set a token string here if your Hugging Face repo is private.

MODEL_FILENAME = "epigranet_embedding_model (1).pt"
EMBEDDINGS_FILENAME = "reference_embeddings.pt"
CLASS_MAPPING_FILENAME = "class_mapping_209 (1).json"

USE_HF_ASSETS = True
MAX_UPLOAD_SIZE_MB = 20


In [ ]:
import subprocess
import sys

if PROJECT_DIR.exists():
    print(f"Using existing project directory: {PROJECT_DIR}")
else:
    if "<your-github-user>" in REPO_URL:
        raise ValueError("Update REPO_URL in the config cell before running setup.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "matplotlib", "pandas"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

print("Colab environment is ready.")


## Download model assets from Hugging Face Hub

If `USE_HF_ASSETS` is `False`, the notebook falls back to the files already committed in the repo.


In [ ]:
from huggingface_hub import hf_hub_download

models_dir = PROJECT_DIR / "models"
models_dir.mkdir(parents=True, exist_ok=True)

def resolve_asset(filename: str, default_parent: Path) -> Path:
    local_path = default_parent / filename
    if not USE_HF_ASSETS:
        if not local_path.exists():
            raise FileNotFoundError(f"Missing local asset: {local_path}")
        return local_path

    if "your-hf-username/" in HF_REPO_ID:
        raise ValueError("Update HF_REPO_ID in the config cell before downloading assets.")

    downloaded = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=filename,
        revision=HF_REVISION,
        token=HF_TOKEN,
    )
    return Path(downloaded)

model_path = resolve_asset(MODEL_FILENAME, models_dir)
embedding_cache_path = resolve_asset(EMBEDDINGS_FILENAME, models_dir)
class_mapping_path = resolve_asset(CLASS_MAPPING_FILENAME, PROJECT_DIR)

print("Resolved assets:")
print(model_path)
print(embedding_cache_path)
print(class_mapping_path)


In [ ]:
import os
import sys

sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)

from pipeline import OCRPredictor, ensure_dir, preprocess_image, segment_characters

RUNTIME_DIR = PROJECT_DIR / "runtime_generated_colab"
UPLOAD_DIR = ensure_dir(RUNTIME_DIR / "uploads")
PREPROCESSED_DIR = ensure_dir(RUNTIME_DIR / "preprocessed")
SEGMENT_DIR = ensure_dir(RUNTIME_DIR / "segments")
BOXED_DIR = ensure_dir(RUNTIME_DIR / "boxed")

predictor = OCRPredictor(
    model_path=model_path,
    class_mapping_path=class_mapping_path,
    embedding_cache_path=embedding_cache_path,
    dataset_path=None,
)

print(f"Model architecture: {predictor.model_arch}")
print(f"Reference classes loaded: {len(predictor.reference_embeddings)}")


## Upload an inscription image

Colab will prompt you to choose a local image file.


In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError("Upload an inscription image to continue.")

uploaded_name = next(iter(uploaded))
input_path = UPLOAD_DIR / uploaded_name
with open(input_path, "wb") as f:
    f.write(uploaded[uploaded_name])

file_size_mb = input_path.stat().st_size / (1024 * 1024)
if file_size_mb > MAX_UPLOAD_SIZE_MB:
    raise ValueError(f"File too large: {file_size_mb:.2f} MB. Limit is {MAX_UPLOAD_SIZE_MB} MB.")

print(f"Saved upload to: {input_path}")


In [ ]:
import pandas as pd

run_id = input_path.stem.replace(" ", "_")
preprocessed_path = PREPROCESSED_DIR / f"{run_id}_preprocessed.png"
boxed_path = BOXED_DIR / f"{run_id}_boxed.png"
roi_dir = ensure_dir(SEGMENT_DIR / run_id)

preprocess_image(input_path, preprocessed_path)
roi_paths = segment_characters(preprocessed_path, roi_dir, boxed_path)
result = predictor.predict_text(roi_paths, preprocessed_path)

segments_used = len(roi_paths) if roi_paths else len(result.tokens)
print(f"Recognized text: {result.text}")
print(f"Confidence: {result.confidence * 100:.2f}%")
print(f"Segments used: {segments_used}")

token_df = pd.DataFrame(result.tokens)
token_df


In [ ]:
import math
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
views = [
    (input_path, "Original"),
    (preprocessed_path, "Preprocessed"),
    (boxed_path, "Segmentation Overlay"),
]

for ax, (path, title) in zip(axes, views):
    img = Image.open(path)
    ax.imshow(img, cmap="gray" if img.mode == "L" else None)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

if roi_paths:
    cols = 6
    rows = math.ceil(len(roi_paths) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.4, rows * 2.4))
    axes_list = list(axes.flat) if hasattr(axes, "flat") else [axes]

    for ax, roi_path in zip(axes_list, roi_paths):
        ax.imshow(Image.open(roi_path))
        ax.set_title(roi_path.stem, fontsize=8)
        ax.axis("off")

    for ax in axes_list[len(roi_paths):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No separate character segments were found. The full preprocessed image was used for prediction.")


In [ ]:
import json

result_path = RUNTIME_DIR / f"{run_id}_result.json"
payload = {
    "input_image": str(input_path),
    "preprocessed_image": str(preprocessed_path),
    "segmentation_overlay_image": str(boxed_path),
    "recognized_text": result.text,
    "confidence": round(result.confidence * 100, 2),
    "num_segments": segments_used,
    "token_predictions": result.tokens,
}

with open(result_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"Saved result summary to: {result_path}")
